# 04. Cruce sector x territorio

Este notebook explora la base complementaria del cuadro `actividad x territorio`.

Importante:

- no forma parte de la base maestra principal
- cubre `2001-2024`
- tiene validación propia
- algunos años quedan en `revisar` u `ok_con_salvedad`


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / 'bases').exists() and (ROOT / 'notebooks').exists():
    project_root = ROOT
else:
    project_root = ROOT.parent

BASE = project_root / 'bases' / 'cruce_sector_territorio'
cross = pd.read_csv(BASE / 'sector_territorio_2001_2024.csv')
coverage = pd.read_csv(BASE / 'cobertura_sector_territorio_1993_2024.csv')
validation = pd.read_csv(BASE / 'validacion_sector_territorio_2001_2024.csv')
validation_summary = pd.read_csv(BASE / 'validacion_sector_territorio_resumen.csv')


## Cobertura

La extracción complementaria solo existe en los años donde el cuadro cruzado fue identificable y legible.


In [ ]:
coverage

## Estado de validación

Aquí se resume si la suma por sectores coincide con el total territorial reportado por el cuadro fuente.


In [ ]:
validation_summary

## Años por estado global

Esta agregación compacta ayuda a distinguir años `ok`, `ok_con_salvedad` y `revisar`.


In [ ]:
year_status = (
    validation_summary.groupby('anio')['estado']
    .agg(lambda values: 'revisar' if 'revisar' in set(values) else ('ok_con_salvedad' if 'ok_con_salvedad' in set(values) else 'ok'))
    .reset_index(name='estado_global')
)
year_status

## Territorios con más huelgas

Se usa el nivel `regional` para evitar mezclar regiones y zonas en un solo ranking.


In [ ]:
regional = cross[cross['nivel_territorial'] == 'regional'].copy()
top_territories = (
    regional.groupby('territorio_homologado_agregado', dropna=False)['huelgas']
    .sum()
    .sort_values(ascending=False)
    .head(12)
    .index
)
pivot_territory = (
    regional[regional['territorio_homologado_agregado'].isin(top_territories)]
    .groupby(['anio', 'territorio_homologado_agregado'], dropna=False)['huelgas']
    .sum()
    .reset_index()
    .pivot(index='anio', columns='territorio_homologado_agregado', values='huelgas')
    .fillna(0)
)
pivot_territory.head()

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(pivot_territory.T, aspect='auto', cmap='YlGnBu')
plt.colorbar(label='Huelgas')
plt.yticks(range(len(pivot_territory.columns)), pivot_territory.columns)
plt.xticks(range(len(pivot_territory.index)), pivot_territory.index, rotation=90)
plt.title('Cruce sector x territorio: territorios regionales con más huelgas')
plt.tight_layout()
plt.show()

## Sectores con más huelgas


In [ ]:
top_sectors = (
    cross.groupby('actividad_homologada_agregada', dropna=False)['huelgas']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)
pivot_sector = (
    cross[cross['actividad_homologada_agregada'].isin(top_sectors)]
    .groupby(['anio', 'actividad_homologada_agregada'], dropna=False)['huelgas']
    .sum()
    .reset_index()
    .pivot(index='anio', columns='actividad_homologada_agregada', values='huelgas')
    .fillna(0)
)
pivot_sector.head()

In [ ]:
plt.figure(figsize=(12, 7))
plt.imshow(pivot_sector.T, aspect='auto', cmap='YlOrRd')
plt.colorbar(label='Huelgas')
plt.yticks(range(len(pivot_sector.columns)), pivot_sector.columns)
plt.xticks(range(len(pivot_sector.index)), pivot_sector.index, rotation=90)
plt.title('Cruce sector x territorio: sectores con más huelgas')
plt.tight_layout()
plt.show()

## Casos con discrepancia de validación

Esta tabla ayuda a revisar manualmente los años y territorios que todavía no cierran exactamente con el total del cuadro fuente.


In [ ]:
validation[validation['estado'] == 'revisar'].sort_values(['anio', 'territorio_original', 'metrica'])